In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import os
os.chdir("/content")

In [3]:
import os
import subprocess
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/vlm-finetuning-project1"
REPO_DIR = "vlm-safety-reasoning"
ENV_PATH = f"{DRIVE_ROOT}/secrets/.env"


def load_secrets(env_path: str) -> dict:
    """Read a .env file and export its values into os.environ."""
    if not os.path.exists(env_path):
        raise FileNotFoundError(f"Secrets file not found at: {env_path}")

    secrets = {}
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            secrets[key] = value.strip(" \"'\r")
            os.environ[key] = secrets[key]
    return secrets


print(">>> Loading secrets...")
secrets = load_secrets(ENV_PATH)
required_keys = ["GIT_EMAIL", "GIT_NAME", "GITHUB_USERNAME", "GITHUB_TOKEN", "HF_TOKEN"]
missing = [k for k in required_keys if k not in secrets]
if missing:
    raise KeyError(f"Missing required secrets: {missing}")
print(">>> Secrets loaded successfully.")

print(">>> Configuring Git identity...")
subprocess.run(["git", "config", "--global", "user.email", secrets["GIT_EMAIL"]], check=True)
subprocess.run(["git", "config", "--global", "user.name", secrets["GIT_NAME"]], check=True)

AUTH_REPO_URL = (
    f"https://{secrets['GITHUB_USERNAME']}:{secrets['GITHUB_TOKEN']}"
    f"@github.com/epmresearch/vlm-safety-reasoning.git"
)

if os.path.exists(REPO_DIR):
    print(">>> Repo already present, pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "remote", "set-url", "origin", AUTH_REPO_URL], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    print(">>> Cloning repo...")
    subprocess.run(["git", "clone", AUTH_REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print(f">>> Working directory: {os.getcwd()}")

print(">>> Copying .env into local workspace...")
shutil.copy(ENV_PATH, ".env")

print(">>> Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(">>> Setup complete.")

>>> Loading secrets...


FileNotFoundError: Secrets file not found at: /content/drive/MyDrive/vlm-finetuning-project1/secrets/.env

In [ ]:
# !rm -rf "/content/vlm-safety-reasoning"

In [ ]:
# import os
# import sys
# import importlib

# base_dir = '/content/' 

# # Go through every folder and file
# for root, dirs, files in os.walk(base_dir):
#     # Skip .git folders and __pycache__
#     if '.git' in dirs:
#         dirs.remove('.git')
#     if '__pycache__' in dirs:
#         dirs.remove('__pycache__')

#     # Build the Python module path
#     # Example: ./my_folder/my_module.py becomes my_folder.my_module
#     module_path = root.replace('/', '.')
    
#     for f in files:
#         if f.endswith('.py') and f != '__init__.py':
#             module_name = f[:-3] # Remove the .py extension
#             full_module_name = f"{module_path}.{module_name}"
            
#             # Remove the leading '.' 
#             if full_module_name.startswith('.'):
#                 full_module_name = full_module_name[1:]

#             # If the module was already imported, reload it
#             if full_module_name in sys.modules:
#                 try:
#                     importlib.reload(sys.modules[full_module_name])
#                     print(f"Reloaded: {full_module_name}")
#                 except Exception as e:
#                     print(f"Could not reload {full_module_name}: {e}")

In [5]:
from data.loader import load_processed_dataset

full_ds = load_processed_dataset()
mini_test = full_ds["test"].select(range(8))
print(mini_test)
print(mini_test.column_names)

2026-07-15 13:06:34 | INFO     | data.loader:load_processed_dataset:183 - Loading fully processed dataset from disk: /content/drive/MyDrive/vlm-finetuning-project1/datasets/processed
2026-07-15 13:06:34 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'train' split: 6308 samples
2026-07-15 13:06:34 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'val' split: 701 samples
2026-07-15 13:06:34 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'test' split: 3004 samples
Dataset({
    features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat'],
    num_rows: 8
})
['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excav

In [6]:
from data.prompt_templates import SYSTEM_PROMPT, UNIFIED_INSPECTION_PROMPT

print("SYSTEM_PROMPT chars:", len(SYSTEM_PROMPT))
print("UNIFIED_PROMPT chars:", len(UNIFIED_INSPECTION_PROMPT))

SYSTEM_PROMPT chars: 309
UNIFIED_PROMPT chars: 1717


In [7]:
from data.preprocessor import build_ground_truth_dict

raw_sample = mini_test[0]
gt = build_ground_truth_dict(raw_sample)

In [8]:
print(raw_sample.keys())
print("=" * 50)
print(gt.keys())

dict_keys(['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat'])
dict_keys(['caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat'])


In [9]:
from models.model_loader import load_model_for_inference

model, tokenizer, model_info = load_model_for_inference(tier="2b")
print("Loaded:", model_info["hf_path"])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
2026-07-15 13:07:02 | INFO     | models.model_loader:load_model_for_inference:154 - Loading model for inference: unsloth/Qwen3-VL-2B-Instruct


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Loaded: unsloth/Qwen3-VL-2B-Instruct


In [15]:
from models.inference import generate_single
import time

sample = mini_test[3]
sample

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=2048x1536>,
 'image_id': '0000007',
 'image_caption': 'Six workers are pouring concrete using a hose at night. The worker at the right is wearing a yellow hard hat while a worker at the left is wearing a red hard hat. The worker with a red hard hat is occluded with another worker with a camouflage suit.',
 'illumination': 'night',
 'camera_distance': 'short distance',
 'view': 'elevation view',
 'quality_of_info': 'poor info',
 'rule_1_violation': {'bounding_box': [[0.14, 0.09, 1.0, 0.66]],
  'reason': 'Multiple workers not wearing hard hats nor high-visibility vests.'},
 'rule_2_violation': None,
 'rule_3_violation': {'bounding_box': [[0.01, 0.68, 0.47, 1.0]],
  'reason': 'Opening not protected on both the left and the right of the images.'},
 'rule_4_violation': None,
 'excavator': [],
 'rebar': [],
 'worker_with_white_hard_hat': []}

In [16]:
t0 = time.time()
raw_output = generate_single(model, tokenizer, sample["image"], max_new_tokens=700)
print(f"Latency: {time.time() - t0:.1f}s")
print(raw_output)

Latency: 75.1s
```json
{
  "caption": "A nighttime construction scene where workers are assembling rebar and concrete formwork. Several workers are on scaffolding, wearing hard hats and work clothes. The site is active, with rebar and formwork visible. The workers are focused on their task, and the area is illuminated by artificial lighting.",
  "rule_1_violation": {
    "reason": "Workers are wearing hard hats and appropriate clothing. However, the workers are not wearing closed-toe shoes, which is a safety requirement. The workers are on a scaffold, and the use of PPE is adequate, but the lack of closed-toe shoes is a violation.",
    "bounding_box": [
      [153, 165, 458, 999]
    ]
  },
  "rule_2_violation": null,
  "rule_3_violation": null,
  "rule_4_violation": null,
  "excavator": [],
  "rebar": [
    [445, 400, 680, 999]
  ],
  "worker_with_white_hard_hat": [
    [153, 165, 458, 999]
  ]
}
```


In [17]:
from evaluation.output_parser import parse_model_output, validate_unified_output

parsed = parse_model_output(raw_output)
print("Valid JSON:", parsed is not None)
if parsed is not None:
    validated = validate_unified_output(parsed)
    print("Schema valid:", validated is not None)
    print(parsed)

Valid JSON: True
Schema valid: True
{'caption': 'A nighttime construction scene where workers are assembling rebar and concrete formwork. Several workers are on scaffolding, wearing hard hats and work clothes. The site is active, with rebar and formwork visible. The workers are focused on their task, and the area is illuminated by artificial lighting.', 'rule_1_violation': {'reason': 'Workers are wearing hard hats and appropriate clothing. However, the workers are not wearing closed-toe shoes, which is a safety requirement. The workers are on a scaffold, and the use of PPE is adequate, but the lack of closed-toe shoes is a violation.', 'bounding_box': [[153, 165, 458, 999]]}, 'rule_2_violation': None, 'rule_3_violation': None, 'rule_4_violation': None, 'excavator': [], 'rebar': [[445, 400, 680, 999]], 'worker_with_white_hard_hat': [[153, 165, 458, 999]]}


Batch on Mini

In [18]:
from models.inference import run_inference

results = run_inference(
    model=model,
    tokenizer=tokenizer,
    dataset=mini_test,
    max_samples=None,   # already only 8 samples
)

Inference: 100%|██████████| 8/8 [05:08<00:00, 38.55s/it]

2026-07-15 13:32:45 | INFO     | models.inference:run_inference:158 - Inference complete: 8 samples processed, 0 errors


In [19]:
print(f"{len(results)} results, {sum('error' in r for r in results)} errors")

8 results, 0 errors


In [27]:
results[2]

{'image_id': '0000005',
 'raw_output': '```json\n{\n  "caption": "A construction site with an orange Doosan excavator parked on a dirt ground. A worker in a red shirt and blue jeans is walking in the background. The excavator is positioned near a pile of sand. The background includes trees, a body of water, and distant buildings. The sky is overcast.",\n  "detected_objects": [\n    [\n      234, 298, 815, 805\n    ],\n    [\n      152, 605, 180, 688\n    ],\n    [\n      152, 605, 180, 688\n    ]\n  ],\n  "rule_1_violation": null,\n  "rule_2_violation": null,\n  "rule_3_violation": null,\n  "rule_4_violation": null,\n  "excavator": [\n    [\n      234, 298, 815, 805\n    ]\n  ],\n  "rebar": [],\n  "worker_with_white_hard_hat": [\n    [\n      152, 605, 180, 688\n    ]\n  ]\n}\n```',
 'sample': {'image_id': '0000005',
  'image_caption': 'An excavator is in the center of the image. There is a person walking on the left, and a mobile crane on the right. The mobile crane is hidden by the e

In [31]:
import json

for i in range(len(results)):
    print(f"\n--- Result {i} ---")
    raw_str = results[i]['raw_output']
    clean_str = raw_str.replace("```json", "").replace("```", "").strip()

    try:
        parsed_json = json.loads(clean_str)
        print(json.dumps(parsed_json, indent=2))
    except json.JSONDecodeError:
        print("Could not parse JSON. Raw Output:")
        print(raw_str)



--- Result 0 ---
{
  "caption": "A large yellow and red mobile crane is parked on a dirt construction site. The crane is positioned on a dirt road, with a pile of rebar and a small building visible in the background. The site is open and undeveloped, with no visible workers or construction activity. The crane is not in operation, and there are no visible safety harnesses, hard hats, or other PPE. The ground is uneven and unpaved, with no visible edge protection or underground work.",
  "detected_objects": [
    {
      "type": "excavator",
      "bounding_box": []
    },
    {
      "type": "rebar",
      "bounding_box": [
        [
          831,
          455,
          1000,
          600
        ]
      ]
    },
    {
      "type": "worker_with_white_hard_hat",
      "bounding_box": []
    }
  ],
  "rule_1_violation": null,
  "rule_2_violation": null,
  "rule_3_violation": null,
  "rule_4_violation": null,
  "excavator": [],
  "rebar": [
    [
      831,
      455,
      1000,
   

Eval on Mini

In [42]:
import sys

for module_name in list(sys.modules.keys()):
    if module_name.startswith("evaluation") or module_name.startswith("data"):
        del sys.modules[module_name]

In [43]:
import transformers
transformers.utils.logging.set_verbosity_error()

# Monkeypatch bert_score's reliance on the missing method
from transformers import RobertaTokenizer
if not hasattr(RobertaTokenizer, "build_inputs_with_special_tokens"):
    RobertaTokenizer.build_inputs_with_special_tokens = lambda self, token_ids_0, token_ids_1=None: [self.cls_token_id] + token_ids_0 + [self.sep_token_id]

In [44]:
from evaluation.evaluator import run_full_evaluation
from data.preprocessor import build_ground_truth_dict

raw_predictions = [r["raw_output"] for r in results]
references = [build_ground_truth_dict(r) for r in mini_test]

eval_out = run_full_evaluation(raw_predictions, references)  # judge_model=None -> reasoning skipped, no Llama load

2026-07-15 14:07:51 | INFO     | evaluation.evaluator:run_full_evaluation:23 - Starting full evaluation pipeline...
2026-07-15 14:07:51 | WARNING  | evaluation.output_parser:parse_model_output:33 - Failed to parse JSON: Expecting ',' delimiter: line 35 column 17 (char 1082)
2026-07-15 14:07:51 | WARNING  | evaluation.output_parser:validate_unified_output:45 - Failed to validate UnifiedOutput schema: 2 validation errors for UnifiedOutput
excavator.0
  Input should be a valid list [type=list_type, input_value={'bounding_box': [265, 478, 425, 555]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
worker_with_white_hard_hat.0
  Input should be a valid list [type=list_type, input_value={'bounding_box': [718, 538, 728, 562]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
2026-07-15 14:07:51 | WARNING  | evaluation.output_parser:validate_unified_output:45 - Failed to validate UnifiedOutput schema

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

2026-07-15 14:07:54 | INFO     | evaluation.evaluator:run_full_evaluation:46 - Computing grounding metrics...
2026-07-15 14:07:54 | INFO     | evaluation.evaluator:run_full_evaluation:50 - Computing safety violation metrics...
2026-07-15 14:07:54 | INFO     | evaluation.evaluator:run_full_evaluation:54 - Computing reasoning metrics (LLM-as-a-Judge)...
2026-07-15 14:07:54 | INFO     | evaluation.evaluator:run_full_evaluation:65 - Evaluation complete.


In [45]:
print(json.dumps(eval_out["metrics"], indent=2))

{
  "json_validity_rate": 0.875,
  "schema_adherence_rate": 0.625,
  "bertscore_precision": 0.858573317527771,
  "bertscore_recall": 0.8793721795082092,
  "bertscore_f1": 0.8685306310653687,
  "meteor": 0.1824346564524755,
  "cider": 0.08995801860687458,
  "grounding_iou_excavator": 0.615570856873823,
  "grounding_iou_rebar": 0.7060350529100529,
  "grounding_iou_worker_with_white_hard_hat": 0.5,
  "grounding_iou_mean": 0.6072019699279586,
  "violation_macro_precision": 1.0,
  "violation_macro_recall": 0.2,
  "violation_macro_f1": 0.33333333333333337,
  "violation_rule_1_precision": 1.0,
  "violation_rule_1_recall": 0.3333333333333333,
  "violation_rule_1_f1": 0.5,
  "violation_rule_2_precision": 0.0,
  "violation_rule_2_recall": 0.0,
  "violation_rule_2_f1": 0.0,
  "violation_rule_3_precision": 0.0,
  "violation_rule_3_recall": 0.0,
  "violation_rule_3_f1": 0.0,
  "violation_rule_4_precision": 0.0,
  "violation_rule_4_recall": 0.0,
  "violation_rule_4_f1": 0.0,
  "violation_grounding_i

In [ ]:
splits = load_dataset_splits()
test_data = splits["test"]
print(f"Test split size: {len(test_data)}")

# Grab a small, deliberately varied set
def pick_diverse_samples(dataset, n_each=1):
    picks = {}
    for sample in dataset:
        for i in range(1, 5):
            key = f"rule_{i}_violation"
            if key not in picks and sample.get(key):
                picks[key] = sample
        if "clean" not in picks and all(sample.get(f"rule_{i}_violation") is None for i in range(1, 5)):
            picks["clean"] = sample
        if len(picks) >= 5:
            break
    return picks

diverse_samples = pick_diverse_samples(test_data)
print("Picked sample categories:", list(diverse_samples.keys()))

2026-07-15 10:38:03 | INFO     | data.loader:load_cleaned_construction_dataset:115 - Loading cleaned dataset from disk: /content/drive/MyDrive/vlm-finetuning-project1/datasets/raw_cleaned
2026-07-15 10:38:36 | INFO     | data.loader:load_cleaned_construction_dataset:119 - Train split size: 7009
2026-07-15 10:38:36 | INFO     | data.loader:load_cleaned_construction_dataset:121 - Test split size: 3004
2026-07-15 10:38:36 | INFO     | data.loader:create_stratified_val_split:50 - Computing strata for balanced train/val split...


Adding strata:   0%|          | 0/7009 [00:00<?, ? examples/s]

In [ ]:
model, tokenizer, model_info = load_model_for_inference(tier="2b")

In [ ]:
import time

test_key = "clean" if "clean" in diverse_samples else list(diverse_samples.keys())[0]
sample = diverse_samples[test_key]

print(f"Testing on sample category: {test_key} | image_id={sample['image_id']}")
sample["image"]  # displays inline in Colab if last expr in cell; otherwise use plt below

start = time.time()
raw_output = generate_single(model, tokenizer, sample["image"], max_new_tokens=MAX_NEW_TOKENS_UNIFIED)
elapsed = time.time() - start

print(f"\n--- Inference took {elapsed:.2f}s ---")
print("\n=== RAW MODEL OUTPUT ===")
print(raw_output)

In [ ]:
parsed = parse_model_output(raw_output)
print("Parsed successfully:", parsed is not None)
if parsed is not None:
    import json
    print(json.dumps(parsed, indent=2))

    validated = validate_unified_output(parsed)
    print("\nSchema valid:", validated is not None)
else:
    print("!! Could not parse JSON — inspect raw_output above for what went wrong.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from data.box_utils import normalize_boxes, scale_1000_to_01

def draw_boxes_on_ax(ax, boxes_01, width, height, color, label):
    for box in boxes_01:
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle(
            (xmin * width, ymin * height),
            (xmax - xmin) * width, (ymax - ymin) * height,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(xmin * width, ymin * height - 5, label, color="white", fontsize=9,
                bbox=dict(facecolor=color, alpha=0.8, edgecolor="none", pad=1))

def compare_prediction_vs_gt(sample, parsed_pred):
    img = sample["image"]
    width, height = img.size
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    for ax, title in zip(axes, ["Ground Truth", "Prediction"]):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title)

    gt_ax, pred_ax = axes
    for cls, color in [("excavator", "blue"), ("rebar", "orange"), ("worker_with_white_hard_hat", "green")]:
        gt_boxes = normalize_boxes(sample.get(cls, []))
        draw_boxes_on_ax(gt_ax, gt_boxes, width, height, color, cls)

    for i in range(1, 5):
        v = sample.get(f"rule_{i}_violation")
        if v and v.get("bounding_box"):
            gt_boxes = normalize_boxes(v["bounding_box"])
            draw_boxes_on_ax(gt_ax, gt_boxes, width, height, "red", f"Rule {i}")

    if parsed_pred:
        for cls, color in [("excavator", "blue"), ("rebar", "orange"), ("worker_with_white_hard_hat", "green")]:
            pred_boxes_1000 = normalize_boxes(parsed_pred.get(cls, []))
            pred_boxes_01 = [scale_1000_to_01(b) for b in pred_boxes_1000]
            draw_boxes_on_ax(pred_ax, pred_boxes_01, width, height, color, cls)

        for i in range(1, 5):
            v = parsed_pred.get(f"rule_{i}_violation")
            if v and v.get("bounding_box"):
                pred_boxes_1000 = normalize_boxes(v["bounding_box"])
                pred_boxes_01 = [scale_1000_to_01(b) for b in pred_boxes_1000]
                draw_boxes_on_ax(pred_ax, pred_boxes_01, width, height, "red", f"Rule {i}")
    else:
        pred_ax.set_title("Prediction (UNPARSEABLE)")

    plt.tight_layout()
    plt.show()

    print("GT caption:", sample.get("image_caption", ""))
    if parsed_pred:
        print("Pred caption:", parsed_pred.get("caption", ""))

compare_prediction_vs_gt(sample, parsed)

In [ ]:
from evaluation.metrics_structural import compute_structural_metrics

results_log = []
raw_outputs_all = []

for key, s in diverse_samples.items():
    print(f"\n{'='*70}\nRunning: {key} | image_id={s['image_id']}\n{'='*70}")
    t0 = time.time()
    raw = generate_single(model, tokenizer, s["image"], max_new_tokens=MAX_NEW_TOKENS_UNIFIED)
    dt = time.time() - t0
    raw_outputs_all.append(raw)

    p = parse_model_output(raw)
    valid_schema = validate_unified_output(p) is not None if p else False

    results_log.append({
        "category": key,
        "image_id": s["image_id"],
        "latency_s": round(dt, 2),
        "json_valid": p is not None,
        "schema_valid": valid_schema,
    })

    print(f"latency={dt:.2f}s | json_valid={p is not None} | schema_valid={valid_schema}")
    compare_prediction_vs_gt(s, p)

import pandas as pd
df_log = pd.DataFrame(results_log)
print("\n=== Sanity Run Summary ===")
print(df_log.to_string(index=False))

struct_metrics = compute_structural_metrics(raw_outputs_all)
print("\n=== Structural Metrics (across this small sample) ===")
print(struct_metrics)

In [ ]:
print("Checklist before running the full baseline notebook:")
print(f"  - JSON validity rate on sample: {struct_metrics.get('json_validity_rate', 0)*100:.0f}%")
print(f"  - Schema adherence rate on sample: {struct_metrics.get('schema_adherence_rate', 0)*100:.0f}%")
print(f"  - Avg latency/sample: {df_log['latency_s'].mean():.2f}s "
      f"-> est. full test set (3004 imgs): {df_log['latency_s'].mean()*3004/60:.1f} min")
print("\nIf JSON/schema validity is low, inspect raw_outputs_all above before burning")
print("compute on the full 3004-sample baseline run.")

### Load Base Model and Dataset

In [ ]:
from core.constants import DEFAULT_MODEL_TIER
from data.loader import load_dataset_splits
from models.model_loader import load_model_for_inference

In [ ]:
splits = load_dataset_splits()

In [ ]:
print(f"Train size: {len(splits['train'])}")
print(f"Validation size: {len(splits['val'])}")
print(f"Test size: {len(splits['test'])}\n")

def print_rule_distribution(split_name, dataset):
    rule_counts = {"rule_1": 0, "rule_2": 0, "rule_3": 0, "rule_4": 0}

    for sample in dataset:
        for i in range(1, 5):
            if sample.get(f"rule_{i}_violation") is not None:
                rule_counts[f"rule_{i}"] += 1

    print(f"--- {split_name} Split Rule Violations ---")
    for rule, count in rule_counts.items():
        percentage = (count / len(dataset)) * 100
        print(f"{rule}: {count} ({percentage:.1f}%)")
    print()

print_rule_distribution("Train", splits["train"])
print_rule_distribution("Validation", splits["val"])

In [ ]:
test_data = splits["test"]

In [ ]:
print(f"Loading baseline model (tier: {DEFAULT_MODEL_TIER})...")
model, tokenizer = load_model_for_inference(tier=DEFAULT_MODEL_TIER)

### Baseline Inference

In [ ]:
from models.inference import run_inference
import json
from core.io import get_drive_path, ensure_dir

# Use a small subset for quick testing (e.g. max_samples=10). Set to None for full evaluation.
MAX_SAMPLES = 50

print(f"Running baseline inference on {MAX_SAMPLES} samples...")
results = run_inference(
    model=model,
    tokenizer=tokenizer,
    dataset=test_data,
    max_samples=MAX_SAMPLES
)

### Cell 5: Evaluate Baseline Outputs

In [ ]:
from evaluation.evaluator import run_full_evaluation

raw_predictions = [res["raw_output"] for res in results]
# Convert test_data (HuggingFace Dataset) to list of dicts for evaluation
references = list(test_data.select(range(MAX_SAMPLES))) if MAX_SAMPLES else list(test_data)

print("Running unified evaluation pipeline...")
eval_results = run_full_evaluation(raw_predictions, references)

# Save results
output_dir = ensure_dir(get_drive_path("results", model_info["short_name"], "baseline"))
with open(output_dir / "metrics.json", "w") as f:
    json.dump(eval_results, f, indent=2)

print(f"Evaluation complete. Results saved to {output_dir}/metrics.json")
print("Metrics Summary:")
print(json.dumps(eval_results["structural"], indent=2))